# 02_cuVS / 00 — Local multimodal embeddings: EmbeddingGemma 2 on vLLM

One open model turns **text, images, audio and video** into vectors in the **same** 768-dim space, so a text query can find a photo, a sound or a video clip. Everything runs on the local GPU: no API keys, no cloud storage.

| | |
|---|---|
| Model | [`google/embeddinggemma-2`](https://huggingface.co/google/embeddinggemma-2): Apache 2.0, not gated, 740M params |
| Output | 768 dims, L2-normalized (MRL: also 512 / 256 / 128) |
| Context | 8,192 tokens. Image ≈ 280 tokens, video ≈ 140 per frame at 1 fps, audio ≈ 25 per second (16 kHz mono) |
| Inference | **vLLM** first. Any modality vLLM can't run on this GPU goes to plain **Hugging Face transformers** (same weights, same vectors) |

**What this notebook does**
1. Install, then check the GPU
2. Load the embedder and decide, per modality, whether vLLM or transformers runs it
3. Cross-modal search check on sample images, sounds and videos
4. Text search, mixed (text + image) inputs, MRL truncation
5. Throughput per modality and backend, saved to `results/`

**T4 notes**
- **float32.** The model must not run in float16 (NaN / wrong vectors), and the T4 has no native bfloat16. Newer GPUs use bfloat16 automatically.
- **vLLM on Turing is not validated for this model.** Earlier runs here showed a dtype mismatch in float32, and the attention kernel needs more shared memory than the T4's 64 KB ([vllm-project/vllm#60847](https://github.com/vllm-project/vllm/issues/60847)). The embedder upcasts the model to float32 and probes each modality right after loading, so any failure shows up in seconds, not halfway through a corpus.

Runtime on a T4: about 5–8 minutes after the install (mostly model download and loading).

## 1. Install

Run once per runtime, then **restart the kernel** and continue from section 2. vLLM replaces the runtime's PyTorch.

`google/embeddinggemma-2` is not gated, so no Hugging Face token is needed. (A token only raises download rate limits: set `os.environ["HF_TOKEN"]` before section 3 if you have one, and never commit it.)

In [ ]:
import importlib.util, subprocess, sys

# transformers >= 5.19 has EmbeddingGemma 2. No -U, so pip keeps a transformers version vLLM accepts.
pkgs = (["vllm==0.32.0"] if importlib.util.find_spec("vllm") is None else []) + [
    "transformers>=5.19", "librosa", "soundfile", "opencv-python-headless", "scikit-image"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("Done. Restart the kernel, then run from section 2.")

## 2. Environment

In [ ]:
import os, gc, json, time, platform, traceback, importlib.util
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

props = torch.cuda.get_device_properties(0)
cc = torch.cuda.get_device_capability(0)

def version(mod):
    try:
        return __import__(mod).__version__
    except Exception as e:
        return f"not installed ({type(e).__name__})"

env = {
    "python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
    "vllm": version("vllm"), "transformers": version("transformers"),
    "gpu": props.name, "gpu_mem_gb": round(props.total_memory / 1e9, 1),
    "compute_capability": f"{cc[0]}.{cc[1]}",
    "shared_mem_per_block_kb": getattr(props, "shared_memory_per_block_optin", 0) // 1024,
    "cpu_cores": os.cpu_count(),
}
pd.Series(env, name="value").to_frame()

## 3. The embedder

**This cell is copied into every notebook in this folder**, so each notebook runs inference itself.

Items are plain dicts. Text carries the model's task prefix; media uses a placeholder token (`<|image|>`, `<|audio|>`, `<|video|>`) and no prefix:

```python
query_item("what causes the aurora")         # {"text": "task: search result | query: ..."}
doc_item("The aurora is ...", title="Aurora")  # {"text": "title: Aurora | text: ..."}
image_item(pil_image)   audio_item("clip.wav")   video_item("clip.mp4")
{"text": "Red shoes <|image|> on wet rock <|video|>", "image": img, "video": "demo.mp4"}   # mixed
```

How it runs:
- **Decoding is parallel.** Audio (resampled to 16 kHz mono) and video (OpenCV) are decoded on all CPU cores while the GPU works.
- **vLLM** gets whole lists at once and schedules the batches itself.
- **transformers** follows the checkpoint's own pipeline (`modules.json`): model → mean pooling over all tokens → L2 normalize. Items are grouped by media type and sorted by length to minimise padding.

In [ ]:
MODEL_ID     = "google/embeddinggemma-2"
BACKEND      = "auto"     # "auto": vLLM where it works, transformers for the rest | "vllm" | "transformers"
DIM          = 768
MAX_LEN      = 8192
VLLM_GPU_MEM = 0.45       # share of GPU memory vLLM may reserve (the rest is for cuVS)
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0) >= (8, 0) else torch.float32   # never float16
BATCH = {"text": 32, "image": 32, "audio": 16, "video": 4}     # transformers batch size per modality

os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")   # engine in this process: real errors, clean teardown

MEDIA = ("image", "audio", "video")

def query_item(q):                return {"text": f"task: search result | query: {q}"}
def doc_item(text, title="none"): return {"text": f"title: {title} | text: {text}"}
def image_item(img):              return {"text": "<|image|>", "image": img}
def audio_item(path):             return {"text": "<|audio|>", "audio": path}
def video_item(path):             return {"text": "<|video|>", "video": path}

def kinds(item):
    return tuple(m for m in MEDIA if m in item) or ("text",)

def normalize(x, dim=DIM):
    x = np.asarray(x, dtype=np.float32)[:, :dim]
    return x / np.linalg.norm(x, axis=1, keepdims=True)

def _decode(item):
    """Load an item's media into memory (runs in a thread pool)."""
    out = dict(item)
    if "audio" in item:
        import librosa
        out["audio"] = librosa.load(item["audio"], sr=16_000, mono=True)[0]
    if "video" in item:
        from transformers.video_utils import load_video
        out["video"] = load_video(item["video"], backend="opencv")          # (frames, metadata)
    return out

def decode_all(items):
    if not any("audio" in it or "video" in it for it in items):
        return list(items)
    with ThreadPoolExecutor(os.cpu_count()) as pool:
        return list(pool.map(_decode, items))

def _short_error(e):
    where = " <- ".join(f"{f.name}:{f.lineno}" for f in traceback.extract_tb(e.__traceback__)[-3:][::-1])
    return f"{type(e).__name__}: {str(e)[:300]}  [{where}]"


class Embedder:
    """google/embeddinggemma-2 on vLLM, with plain transformers for any modality vLLM can't run here."""

    def __init__(self, backend=BACKEND):
        self.vllm_kinds, self.errors, self.llm, self.hf = set(), {}, None, None
        if backend in ("auto", "vllm") and importlib.util.find_spec("vllm"):
            self._start_vllm()
        if backend == "vllm" and not self.vllm_kinds:
            raise RuntimeError(f"vLLM is not usable on this GPU: {self.errors}")
        self.routes = {k: "vllm" if k in self.vllm_kinds else "transformers" for k in ("text", *MEDIA)}
        if "transformers" in self.routes.values():
            self._start_hf()
        print(f"[{str(DTYPE).replace('torch.', '')}] routes: {self.routes}")

    # ---- vLLM ----------------------------------------------------------------
    def _start_vllm(self):
        from vllm import LLM
        probes, wanted = self._probes(), {"text", *MEDIA}
        while True:
            t0 = time.perf_counter()
            try:
                self.llm = LLM(model=MODEL_ID, runner="pooling", enforce_eager=True,
                               dtype="bfloat16" if DTYPE == torch.bfloat16 else "float32",
                               max_model_len=MAX_LEN, gpu_memory_utilization=VLLM_GPU_MEM,
                               limit_mm_per_prompt={m: (4 if m == "image" else 1) if m in wanted else 0 for m in MEDIA})
                if DTYPE == torch.float32:
                    self.llm.apply_model(lambda model: model.float())   # some checkpoint parts stay bfloat16 otherwise
            except Exception as e:
                self.errors["start"] = _short_error(e)
                print("vLLM did not start:", self.errors["start"])
                return self._stop_vllm()
            failed = next((k for k in ["text", *[m for m in MEDIA if m in wanted]] if not self._probe_ok(probes[k], k)), None)
            if failed is None:
                self.vllm_kinds = wanted
                print(f"vLLM ready for {sorted(wanted)} in {time.perf_counter() - t0:.0f} s")
                return
            self._stop_vllm()                    # engine state is unknown after a failed request: rebuild
            if failed == "text":
                return
            wanted.discard(failed)               # next attempt skips that modality's encoder

    def _probe_ok(self, item, kind):
        try:
            self._embed_vllm([item])
            return True
        except Exception as e:
            self.errors[kind] = _short_error(e)
            print(f"vLLM failed on {kind}: {self.errors[kind]}")
            return False

    @staticmethod
    def _probes():
        """Tiny item per modality, so kernel or dtype problems surface at load time."""
        import cv2, soundfile as sf
        from PIL import Image
        os.makedirs("assets", exist_ok=True)
        sf.write("assets/_probe.wav", np.zeros(16_000, np.float32), 16_000)
        vw = cv2.VideoWriter("assets/_probe.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 2, (64, 64))
        for i in range(4):
            vw.write(np.full((64, 64, 3), 60 * i, np.uint8))
        vw.release()
        return {"text": doc_item("probe " * 600), "image": image_item(Image.new("RGB", (64, 64), "red")),
                "audio": audio_item("assets/_probe.wav"), "video": video_item("assets/_probe.mp4")}

    def _stop_vllm(self):
        self.llm = None
        try:
            from vllm.distributed.parallel_state import destroy_model_parallel, destroy_distributed_environment
            destroy_model_parallel(); destroy_distributed_environment()
        except Exception:
            pass
        gc.collect(); torch.cuda.empty_cache()

    def _embed_vllm(self, items, chunk=512):
        out = []
        for i in range(0, len(items), chunk):           # bound host memory used by decoded media
            prompts = []
            for it in decode_all(items[i:i + chunk]):
                mm = {}
                if "image" in it: mm["image"] = it["image"]
                if "audio" in it: mm["audio"] = (it["audio"], 16_000)
                if "video" in it: mm["video"] = it["video"]
                prompts.append({"prompt": it["text"], "multi_modal_data": mm} if mm else it["text"])
            out += [o.outputs.embedding for o in self.llm.embed(prompts, use_tqdm=False)]
        return np.asarray(out, dtype=np.float32)

    # ---- transformers ----------------------------------------------------------
    def _start_hf(self):
        from transformers import AutoModel, AutoProcessor
        t0 = time.perf_counter()
        self.proc = AutoProcessor.from_pretrained(MODEL_ID)
        self.hf = AutoModel.from_pretrained(MODEL_ID, dtype=DTYPE).to("cuda").eval()
        print(f"transformers ready in {time.perf_counter() - t0:.0f} s")

    @torch.inference_mode()
    def _hf_batch(self, items):
        kw = {"text": [it["text"] for it in items], "return_tensors": "pt", "padding": True}
        if "image" in items[0]:
            kw["images"] = [it["image"] if isinstance(it["image"], list) else [it["image"]] for it in items]
        if "audio" in items[0]:
            kw["audio"], kw["sampling_rate"] = [it["audio"] for it in items], 16_000
        if "video" in items[0]:
            kw["videos"], kw["video_metadata"] = [it["video"][0] for it in items], [it["video"][1] for it in items]
        batch = {k: v.to("cuda", DTYPE) if v.is_floating_point() else v.to("cuda")
                 for k, v in self.proc(**kw).items() if torch.is_tensor(v)}
        hidden = self.hf(**batch).last_hidden_state                       # (batch, tokens, 768)
        mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        return ((hidden * mask).sum(1) / mask.sum(1)).float().cpu().numpy()   # mean pooling

    def _embed_hf(self, items):
        out = np.zeros((len(items), DIM), np.float32)
        groups = {}
        for i, it in enumerate(items):
            groups.setdefault(kinds(it), []).append(i)
        for kind, idx in groups.items():
            idx.sort(key=lambda i: len(items[i]["text"]))                 # similar lengths -> less padding
            bs = BATCH[kind[0]]
            for j in range(0, len(idx), bs):
                part = idx[j:j + bs]
                out[part] = self._hf_batch(decode_all([items[i] for i in part]))
        return out

    # ---- public --------------------------------------------------------------
    def embed(self, items, dim=DIM):
        """List of item dicts -> float32 array (len(items), dim), L2-normalized."""
        out = np.zeros((len(items), DIM), np.float32)
        on_vllm = [set(kinds(it)) <= self.vllm_kinds for it in items]
        iv = [i for i, v in enumerate(on_vllm) if v]
        ih = [i for i, v in enumerate(on_vllm) if not v]
        if iv: out[iv] = self._embed_vllm([items[i] for i in iv])
        if ih: out[ih] = self._embed_hf([items[i] for i in ih])
        return normalize(out, dim)

    def close(self):
        self._stop_vllm()
        self.hf = None
        gc.collect(); torch.cuda.empty_cache()


emb = Embedder()

## 4. Sample media

Small local samples (the real datasets come in notebook 01):
- **Images:** four photos bundled with scikit-image
- **Audio:** four 15 s clips from librosa's examples: whale song, trumpet, speech, drums and bass
- **Video:** two 8 s clips generated here: a bouncing red ball, and a slow zoom into the cat photo

In [ ]:
import cv2, soundfile as sf, librosa
from PIL import Image
from skimage import data as sk

os.makedirs("assets", exist_ok=True)

images = {"an astronaut": sk.astronaut(), "a cat": sk.chelsea(), "a cup of coffee": sk.coffee(), "a rocket": sk.rocket()}
images = {k: Image.fromarray(v) for k, v in images.items()}

audios = {}
for key, label in {"humpback": "whale song", "trumpet": "trumpet", "libri1": "speech", "choice": "drums and bass"}.items():
    path = f"assets/{key}.wav"
    if not os.path.exists(path):
        sf.write(path, librosa.load(librosa.ex(key), sr=16_000, mono=True, duration=15)[0], 16_000)
    audios[label] = path

def write_video(path, frames, fps=10):
    h, w = frames[0].shape[:2]
    vw = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for f in frames:
        vw.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
    vw.release()

if not os.path.exists("assets/red_ball.mp4"):
    frames = []
    for t in range(80):
        f = np.full((240, 320, 3), 235, np.uint8)
        cv2.line(f, (0, 220), (320, 220), (90, 90, 90), 3)
        cv2.circle(f, (40 + 3 * t, int(200 - 160 * abs(np.sin(t / 8)))), 18, (220, 30, 30), -1)
        frames.append(f)
    write_video("assets/red_ball.mp4", frames)
if not os.path.exists("assets/cat_zoom.mp4"):
    cat, frames = sk.chelsea(), []
    H, W = cat.shape[:2]
    for t in range(80):
        s = 1 - 0.5 * t / 80
        h, w = int(H * s), int(W * s)
        frames.append(cv2.resize(cat[(H - h) // 2:(H + h) // 2, (W - w) // 2:(W + w) // 2], (320, 240)))
    write_video("assets/cat_zoom.mp4", frames)
videos = {"a red ball bouncing": "assets/red_ball.mp4", "a cat (zooming in)": "assets/cat_zoom.mp4"}

print(f"{len(images)} images, {len(audios)} audio clips, {len(videos)} videos")

## 5. Cross-modal search

Ten **text** queries against one pool of 10 media items (4 images, 4 sounds, 2 videos). Each query has exactly one correct item, in a different modality.

In [ ]:
pool   = [image_item(v) for v in images.values()] + [audio_item(v) for v in audios.values()] + [video_item(v) for v in videos.values()]
labels = [f"[image] {k}" for k in images] + [f"[audio] {k}" for k in audios] + [f"[video] {k}" for k in videos]
queries = ["an astronaut in a space suit", "a cat", "a cup of coffee", "a rocket launch",
           "a whale singing underwater", "a trumpet playing", "a person reading a book aloud",
           "drums and bass music", "a red ball bouncing on the floor", "a close-up of a cat"]

t0 = time.perf_counter()
P = emb.embed(pool)
Q = emb.embed([query_item(q) for q in queries])
print(f"embedded {len(pool)} media items + {len(queries)} queries in {time.perf_counter() - t0:.1f} s")

S = Q @ P.T
top1 = int((S.argmax(1) == np.arange(len(queries))).sum())
print(f"top-1 correct: {top1} / {len(queries)}\n")
for i, (q, row) in enumerate(zip(queries, S)):
    print(f"{'ok' if row.argmax() == i else '--'}  {q:<34} -> {labels[row.argmax()]:<30} {row.max():.3f}")

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(S, cmap="viridis")
ax.set_xticks(range(len(labels)), labels, rotation=40, ha="right", fontsize=8)
ax.set_yticks(range(len(queries)), queries, fontsize=8)
for i, j in enumerate(S.argmax(1)):
    ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="white", lw=2))
fig.colorbar(im, ax=ax, label="cosine similarity")
ax.set_title(f"Text queries vs images, audio and video  (top-1 {top1}/{len(queries)})")
plt.tight_layout(); plt.show()

## 6. Text search, mixed inputs and MRL

In [ ]:
docs = [("Aurora", "The northern lights are caused by charged particles from the sun colliding with gases in the atmosphere."),
        ("Turing (microarchitecture)", "Turing is an NVIDIA GPU architecture used in the Tesla T4 and the GeForce RTX 20 series."),
        ("Mango", "The mango is an edible stone fruit produced by the tropical tree Mangifera indica.")]
text_q = ["what causes the aurora borealis", "which gpu architecture does the T4 use", "tropical fruit"]
D = emb.embed([doc_item(t, title) for title, t in docs])
print("Text search (rows: queries, cols: docs)")
print(pd.DataFrame(emb.embed([query_item(q) for q in text_q]) @ D.T, index=text_q, columns=[d[0] for d in docs]).round(3), "\n")

mixed = emb.embed([{"text": "Space mission photo: <|image|> crew member ready for launch", "image": images["an astronaut"]},
                   {"text": "Breakfast table: <|image|> freshly brewed", "image": images["a cup of coffee"]}])
mix_q = ["astronaut preparing for a space launch", "morning coffee"]
print("Mixed text + image items")
print(pd.DataFrame(emb.embed([query_item(q) for q in mix_q]) @ mixed.T, index=mix_q, columns=["astronaut post", "coffee post"]).round(3), "\n")

print("MRL: shorter vectors (re-normalized), same top-1 as 768 dims?")
for d in [512, 256, 128]:
    agree = int(((normalize(Q, d) @ normalize(P, d).T).argmax(1) == S.argmax(1)).sum())
    print(f"  {d:>3} dims: {agree}/{len(queries)} queries agree, {d * 4:>4} bytes per vector ({768 / d:.1f}x smaller)")

## 7. Throughput

Items per second for each modality on this GPU (one warmup, then a timed run). Text uses ~500-token passages, the chunk size planned for notebook 01. If vLLM handles any modality, plain transformers is measured as well, and the two backends' vectors are compared.

In [ ]:
passage = " ".join(("Graphics processors run thousands of threads in parallel, which makes them well suited to "
                    "embedding large collections of documents, images, audio and video. ").split() * 20)[:2600]
bench = {
    "text": [doc_item(f"{passage} {i}", "Benchmark") for i in range(128)],
    "image": [image_item(list(images.values())[i % 4]) for i in range(64)],
    "audio": [audio_item(list(audios.values())[i % 4]) for i in range(32)],
    "video": [video_item(list(videos.values())[i % 2]) for i in range(16)],
}

def throughput(e, items):
    e.embed(items[:4]); torch.cuda.synchronize()
    t0 = time.perf_counter(); e.embed(items); torch.cuda.synchronize()
    return len(items) / (time.perf_counter() - t0)

label = lambda e: "vLLM" if set(e.routes.values()) == {"vllm"} else "transformers" if set(e.routes.values()) == {"transformers"} else "vLLM + transformers"
tput = {label(emb): {k: throughput(emb, v) for k, v in bench.items()}}

if emb.vllm_kinds:
    hf_only = Embedder("transformers")
    tput["transformers"] = {k: throughput(hf_only, v) for k, v in bench.items()}
    for k in sorted(emb.vllm_kinds):
        cos = np.diag(emb.embed(bench[k][:4]) @ hf_only.embed(bench[k][:4]).T)
        print(f"vLLM vs transformers cosine on {k:<5}: min {cos.min():.4f}")
    hf_only.close(); del hf_only

tp = pd.DataFrame(tput).round(1)
tp.index.name = "items / s"
display(tp)

ax = tp.plot.barh(figsize=(8, 3), logx=True)
ax.set_xlabel("items per second (log, higher is better)")
ax.set_title(f"EmbeddingGemma 2 on {props.name} ({str(DTYPE).replace('torch.', '')})")
plt.tight_layout(); plt.show()

per_hour = {k: int(v * 3600) for k, v in tp.iloc[:, 0].items()}
print("At this speed, one hour embeds:", per_hour)

In [ ]:
os.makedirs("results", exist_ok=True)
report = {"env": env, "dtype": str(DTYPE), "routes": emb.routes, "vllm_errors": emb.errors,
          "cross_modal_top1": f"{top1}/{len(queries)}", "throughput_items_per_s": tput}
with open("results/00_embedder_check.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps({k: report[k] for k in ["dtype", "routes", "cross_modal_top1"]}, indent=2))

## Summary

- **EmbeddingGemma 2 runs locally** and puts text, images, audio and video in one 768-dim space. Section 5 checks that a text query lands on the right image, sound or video.
- **Routing.** The `routes` line shows which modalities run on vLLM and which on plain transformers on this GPU. Both use the same weights, so vectors can be mixed in one index.
- **MRL.** Vectors can be cut to 512 / 256 / 128 dims (re-normalized) to save memory; notebooks 03–06 measure what that does to search quality and speed.
- **Throughput** sets how large a corpus notebook 01 can embed in reasonable time.

**Next:** `01_build_multimodal_corpus.ipynb` — real datasets for every modality, embedded and saved locally.